In [1]:
from typing import List, Sequence
from autogen_agentchat.teams import SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.ui import Console
from autogen_agentchat.agents import UserProxyAgent, AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
import os 
import sys
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()

True

This system uses three specialized agents:

Planning Agent: The strategic coordinator that breaks down complex tasks into manageable subtasks.

Web Search Agent: An information retrieval specialist that interfaces with the web_search_tool.

Data Analyst Agent: An agent specialist in performing calculations equipped with percentage_change_tool.

The tools search_web_tool and percentage_change_tool are external tools that the agents can use to perform their tasks.

In [2]:
model_client = OpenAIChatCompletionClient(
    model='gpt-3.5-turbo'
)

In [3]:
#mock tool
def web_search_tool(query:str) -> str:
    if "2007-2008" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."

In [4]:
def percentage_change_tool(start:float, end: float)-> float:
    return ((end - start) / start) * 100

Now specialized agents are created using the AssistantAgent class. It is important to note that the agents’ name and description attributes are used by the model to determine the next speaker, so it is recommended to provide meaningful names and descriptions.

In [5]:
planning_agent = AssistantAgent(
    "PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Perform calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

In [6]:
search_web_agent = AssistantAgent(
    "SearchWebAgent",
    description="An agent for searching information on the web.",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""
    You are a web search agent.
    Your only tool is search_tool - use it to find information.
    You make only one search call at a time.
    Once you have the results, you never do calculations based on them.
    """,
)

In [7]:
data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""
    You are a data analyst.
    Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.
    If you have not seen the data, ask for it.
    """,
)

By default, AssistantAgent returns the tool output as the response. If your tool does not return a well-formed string in natural language format, you may want to add a reflection step within the agent by setting reflect_on_tool_use=True when creating the agent. This will allow the agent to reflect on the tool output and provide a natural language response.

Workflow ->
The task is received by the SelectorGroupChat which, based on agent descriptions, selects the most appropriate agent to handle the initial task (typically the Planning Agent).

The Planning Agent analyzes the task and breaks it down into subtasks, assigning each to the most appropriate agent using the format: <agent> : <task>

Based on the conversation context and agent descriptions, the SelectorGroupChat manager dynamically selects the next agent to handle their assigned subtask.

The Web Search Agent performs searches one at a time, storing results in the shared conversation history.

The Data Analyst processes the gathered information using available calculation tools when selected.

The workflow continues with agents being dynamically selected until either:

The Planning Agent determines all subtasks are complete and sends “TERMINATE”

An alternative termination condition is met (e.g., a maximum number of messages)

When defining your agents, make sure to include a helpful description since this is used to decide which agent to select next.

Termination Conditions ->
Let’s use two termination conditions: TextMentionTermination to end the conversation when the Planning Agent sends “TERMINATE”, and MaxMessageTermination to limit the conversation to 25 messages to avoid infinite loop.

In [8]:
text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=10)
termination = text_mention_termination | max_messages_termination


SelectorGroupChat uses a model to select the next speaker based on the conversation context. We will use a custom selector prompt to properly align with the workflow.

In [9]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""


Running Team

In [10]:
team = SelectorGroupChat(
    [planning_agent, search_web_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,  # Allow an agent to speak multiple turns in a row.
)


In [11]:
task = "Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"


In [12]:
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To find out who was the Miami Heat player with the highest points in the 2006-2007 season, we will need to assign tasks to our team members. 

1. WebSearchAgent: Find the Miami Heat player with the highest points in the 2006-2007 season.
2. DataAnalystAgent: Calculate the percentage change in total rebounds of that player between the 2007-2008 and 2008-2009 seasons.

Let's get started. 

1. WebSearchAgent: Please find the Miami Heat player with the highest points in the 2006-2007 season.
2. DataAnalystAgent: Once we have that information, please calculate the percentage change in total rebounds of that player between the 2007-2008 and 2008-2009 seasons.

Thank you for your help!

TERRMINATE
---------- ToolCallRequestEv

TaskResult(messages=[TextMessage(id='33e47640-ca08-40c8-818c-d9c4c1a02451', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 15, 12, 51, 38, 740522, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='6f49b722-e478-4f68-b8ee-773bc171f0ec', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=161, completion_tokens=177), metadata={}, created_at=datetime.datetime(2026, 6, 15, 12, 51, 45, 532581, tzinfo=datetime.timezone.utc), content="To find out who was the Miami Heat player with the highest points in the 2006-2007 season, we will need to assign tasks to our team members. \n\n1. WebSearchAgent: Find the Miami Heat player with the highest points in the 2006-2007 season.\n2. DataAnalystAgent: Calculate the percentage change in total rebou